# Post-Step Consolidation

## What you'll learn

- Use `post_step` to chain a curator after a creator step
- Consolidate per-worker appendable files into a single file
- How consolidated artifacts get new content-addressed IDs

**Prerequisites:** [External File Storage](11-external-file-storage.ipynb)  
**Estimated time:** 10 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from artisan.operations.curator import ConsolidateAppendables
from artisan.operations.examples import AppendableGenerator
from artisan.orchestration import Backend, PipelineManager
from artisan.storage.core.artifact_store import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

## The problem

When a creator step runs on multiple workers (e.g., with SLURM),
each worker writes its own output files. For appendable artifacts, this
means N workers produce N separate JSONL files. Downstream operations
that want to process all records need a single combined file.

You could write a separate pipeline step to merge the files, but
this is boilerplate that follows the same pattern every time.
The `post_step` parameter automates it: pass a curator operation,
and the framework auto-inserts it after the main step.

In [ ]:
env = tutorial_setup("post_step_consolidation")
DELTA_ROOT = env.delta_root
FILES_ROOT = DELTA_ROOT.parent / "files"

pipeline = PipelineManager.create(
    name="consolidation_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

## Running with `post_step`

Pass `post_step=ConsolidateAppendables` to `run()`. The generator
uses `num_files=3` to split its output across three JSONL files
(simulating multi-worker output). The framework then automatically
wires the generator's outputs as inputs to the consolidation curator.

In [ ]:
step = pipeline.run(
    operation=AppendableGenerator,
    name="generate",
    params={"count": 15, "num_files": 3, "seed": 42},
    backend=Backend.LOCAL,
    post_step=ConsolidateAppendables,
)

print(f"Returned step: {step.step_name}")
print(f"Output roles:  {step.output_roles}")

In [ ]:
inspect_pipeline(DELTA_ROOT)

Two steps appear in the pipeline:

- **`generate`** — the `AppendableGenerator` that wrote three
  JSONL files (one per `num_files`)
- **`generate.post`** — the `ConsolidateAppendables` curator that
  merged them into one file

The `post_step` name is always `{name}.post`. The returned `StepResult`
is from the post_step, not the generator — downstream steps that
wire from this result get the consolidated artifacts.

## The files_root layout

The generator writes its JSONL files under its step number. The
consolidation curator writes the combined file under its own step
number.

In [ ]:
def show_tree(path: Path, prefix: str = "", max_depth: int = 3, depth: int = 0) -> None:
    """Display a directory tree up to max_depth."""
    if depth >= max_depth or not path.is_dir():
        return
    entries = sorted(path.iterdir())
    for i, entry in enumerate(entries):
        connector = (
            "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        )
        suffix = "/" if entry.is_dir() else ""
        print(f"{prefix}{connector}{entry.name}{suffix}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            show_tree(entry, prefix + extension, max_depth, depth + 1)


print(f"{FILES_ROOT.name}/")
show_tree(FILES_ROOT, max_depth=4)

### The combined file

The consolidated `combined.jsonl` contains all 15 records from the
three per-file JSONL files, concatenated in sorted file order.

In [ ]:
# Find the combined file
combined_files = list(FILES_ROOT.rglob("combined.jsonl"))
combined_path = combined_files[0]

lines = combined_path.read_text().strip().split("\n")
print(f"Combined file: {len(lines)} records")
print()
for line in lines[:3]:
    record = json.loads(line)
    print(f"  {record['record_id']}: {record['values']}")
if len(lines) > 3:
    print(f"  ... ({len(lines) - 3} more)")

## Consolidated artifacts

The consolidation curator produces new `AppendableArtifact`s that
point to the combined file. Each preserves its `record_id` and
`content_hash` from the original.

In [ ]:
store = ArtifactStore(DELTA_ROOT)

# Load artifacts from both steps
all_appendable_ids = sorted(store.load_artifact_ids_by_type("appendable"))
all_appendables = store.get_artifacts_by_type(all_appendable_ids, "appendable")

# Group by step
by_step: dict[int, list] = {}
for art in all_appendables.values():
    by_step.setdefault(art.origin_step_number, []).append(art)

for step_num in sorted(by_step):
    arts = by_step[step_num]
    paths = sorted({a.external_path for a in arts})
    print(f"Step {step_num}: {len(arts)} artifacts, {len(paths)} file(s)")
    for p in paths:
        print(f"  path: .../{Path(p).name}")

### New artifact IDs

Because `external_path` is part of the content hash, consolidated
artifacts get different `artifact_id`s than the originals — even
though the record content is identical. This preserves full
provenance: you can trace each consolidated artifact back to its
per-worker source.

In [ ]:
# Pick the first record_id and compare across steps
for step_num in sorted(by_step):
    art = next(a for a in by_step[step_num] if a.record_id == "rec_000000")
    print(
        f"Step {step_num}: id={art.artifact_id[:16]}...  path=.../{Path(art.external_path).name}"
    )

::::{note}
Here we use `num_files=3` to simulate multi-worker output within a
single `Backend.LOCAL` run. With SLURM or other parallel backends,
each worker naturally writes its own file — the consolidation pattern
is the same either way.
::::

In [ ]:
result = pipeline.finalize()
print(
    f"Pipeline complete: {result['total_steps']} steps, success={result['overall_success']}"
)

## Summary

- **`post_step`** auto-inserts a curator after a creator step,
  wiring the creator's outputs as the curator's inputs
- **`ConsolidateAppendables`** concatenates per-worker JSONL files
  into a single `combined.jsonl`
- Consolidated artifacts get **new `artifact_id`s** because
  `external_path` is part of the content hash
- The returned `StepResult` is from the post_step — downstream
  steps get the consolidated artifacts

## Next steps

- [SLURM Execution](07-slurm-execution.ipynb) —
  Run on HPC clusters where consolidation merges real multi-worker output
- [Provenance Graphs](../analysis/01-provenance-graphs.ipynb) —
  Visualize the lineage from per-worker to consolidated artifacts
- [External File Storage](11-external-file-storage.ipynb) —
  The two external-content patterns in detail